# MazuTalk Post-Training (Colab GPU)

QLoRA SFT + 소규모 DPO 를 Colab GPU에서 실행한다. 계획서: `docs/LLM_POST_TRAINING_PLAN.md`.

**전제**
- 런타임: GPU (T4 16GB 이상 권장). Runtime > Change runtime type > GPU.
- 데이터 파일(`ai/data/processed/sft_train.jsonl`, `dpo_train.jsonl`, `eval_set.jsonl`)은 로컬에서 생성해 repo에 포함하거나 업로드.
- `ai/configs/model.yaml` 의 `base_model` 을 실제 HF 체크포인트로 확정.

역할 분담: **데이터 생성/baseline = 로컬(Ollama)**, **학습/추론/평가 = Colab(GPU)**.

In [ ]:
# 1. GPU 확인
!nvidia-smi

In [ ]:
# 2. 저장소 가져오기 (방법 A: git clone / 방법 B: 직접 업로드)
# !git clone https://github.com/Mazu-Talk/MazuTalk.git MazuTalk
# %cd MazuTalk
import os; print('cwd:', os.getcwd())

In [ ]:
# 3. Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mazutalk/models

In [ ]:
# 4. 학습 의존성 설치
!pip install -q -r ai/requirements-train.txt

In [ ]:
# 5. Hugging Face 로그인 (gated 모델 접근 시)
from huggingface_hub import login
login()  # HF 토큰 입력

In [ ]:
# 6. base_model 확인 (필요시 수정)
!cat ai/configs/model.yaml

## Phase 2. QLoRA SFT

In [ ]:
!python ai/scripts/train_sft.py \
    --model-config ai/configs/model.yaml \
    --train-config ai/configs/sft.yaml \
    --data ai/data/processed/sft_train.jsonl \
    --out ai/models/sft

!mkdir -p /content/drive/MyDrive/mazutalk/models/sft
!cp -r ai/models/sft/. /content/drive/MyDrive/mazutalk/models/sft/

## SFT 모델 평가 (frozen eval set)

In [ ]:
!python ai/scripts/infer_hf.py --adapter ai/models/sft --out ai/data/processed/sft_outputs.jsonl
!python ai/scripts/evaluate.py --in ai/data/processed/sft_outputs.jsonl --csv ai/data/processed/sft_eval.csv

## Phase 4. DPO (SFT 어댑터에서 출발)

In [ ]:
!python ai/scripts/train_dpo.py \
    --model-config ai/configs/model.yaml \
    --train-config ai/configs/dpo.yaml \
    --data ai/data/processed/dpo_train.jsonl \
    --sft-adapter ai/models/sft \
    --out ai/models/dpo

!mkdir -p /content/drive/MyDrive/mazutalk/models/dpo
!cp -r ai/models/dpo/. /content/drive/MyDrive/mazutalk/models/dpo/

In [ ]:
!python ai/scripts/infer_hf.py --adapter ai/models/dpo --out ai/data/processed/dpo_outputs.jsonl
!python ai/scripts/evaluate.py --in ai/data/processed/dpo_outputs.jsonl --csv ai/data/processed/dpo_eval.csv

## Phase 5. 병합 + GGUF + Modelfile

In [ ]:
# llama.cpp 준비(GGUF 변환용)
!git clone https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python ai/scripts/export_gguf.py --adapter ai/models/dpo --llama-cpp llama.cpp --quant Q4_K_M

!mkdir -p /content/drive/MyDrive/mazutalk/models/merged /content/drive/MyDrive/mazutalk/models/gguf
!cp -r ai/models/merged/. /content/drive/MyDrive/mazutalk/models/merged/
!cp -r ai/models/gguf/. /content/drive/MyDrive/mazutalk/models/gguf/

In [ ]:
# 결과물 다운로드 (대용량 merged 모델은 Drive 백업만 사용)
!zip -r mazutalk_models.zip ai/models/dpo ai/models/gguf ai/data/processed/*_eval.csv
from google.colab import files; files.download('mazutalk_models.zip')